In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, SimpleRNN, Embedding, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, History
import time

# ============================================================================
# ЧАСТЬ 1: МНОГОСЛОЙНАЯ НЕЙРОННАЯ СЕТЬ (MLP) - САМОСТОЯТЕЛЬНАЯ РЕАЛИЗАЦИЯ
# ============================================================================

In [ ]:
class NeuralNetwork:    
    def __init__(self, layer_sizes, learning_rate=0.01, activation='relu'):
        """
        layer_sizes: список с размерами слоев, например [input_size, 128, 64, output_size]
        """
        self.layer_sizes = layer_sizes
        self.learning_rate = learning_rate
        self.activation_type = activation
        
        # Инициализация весов и смещений
        self.weights = []
        self.biases = []
        
        for i in range(len(layer_sizes) - 1):
            # Xavier/Glorot инициализация
            weight = np.random.randn(layer_sizes[i], layer_sizes[i+1]) * np.sqrt(2.0 / layer_sizes[i])
            bias = np.zeros((1, layer_sizes[i+1]))
            
            self.weights.append(weight)
            self.biases.append(bias)
        
        # Для хранения истории обучения
        self.train_losses = []
        self.val_losses = []
        self.train_accuracies = []
        self.val_accuracies = []
    
    def relu(self, x):
        """ReLU активация"""
        return np.maximum(0, x)
    
    def relu_derivative(self, x):
        """Производная ReLU"""
        return (x > 0).astype(float)
    
    def sigmoid(self, x):
        """Sigmoid активация"""
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    def sigmoid_derivative(self, x):
        """Производная Sigmoid"""
        s = self.sigmoid(x)
        return s * (1 - s)
    
    def softmax(self, x):
        """Softmax для выходного слоя"""
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)
    
    def activation(self, x):
        """Применение активации"""
        if self.activation_type == 'relu':
            return self.relu(x)
        elif self.activation_type == 'sigmoid':
            return self.sigmoid(x)
        return x
    
    def activation_derivative(self, x):
        """Производная активации"""
        if self.activation_type == 'relu':
            return self.relu_derivative(x)
        elif self.activation_type == 'sigmoid':
            return self.sigmoid_derivative(x)
        return np.ones_like(x)
    
    def forward_propagation(self, X):
        """Прямое распространение"""
        self.z_values = []  # Значения до активации
        self.a_values = [X]  # Значения после активации
        
        A = X
        
        # Проход через скрытые слои
        for i in range(len(self.weights) - 1):
            Z = np.dot(A, self.weights[i]) + self.biases[i]
            A = self.activation(Z)
            
            self.z_values.append(Z)
            self.a_values.append(A)
        
        # Выходной слой с softmax
        Z = np.dot(A, self.weights[-1]) + self.biases[-1]
        A = self.softmax(Z)
        
        self.z_values.append(Z)
        self.a_values.append(A)
        
        return A
    
    def compute_loss(self, y_true, y_pred):
        """Кросс-энтропия"""
        m = y_true.shape[0]
        log_likelihood = -np.log(y_pred[range(m), y_true] + 1e-7)
        loss = np.sum(log_likelihood) / m
        return loss
    
    def backward_propagation(self, X, y_true):
        """Обратное распространение ошибки"""
        m = X.shape[0]
        
        # Градиенты
        dW = [None] * len(self.weights)
        db = [None] * len(self.biases)
        
        # Ошибка выходного слоя
        y_one_hot = np.zeros((m, self.layer_sizes[-1]))
        y_one_hot[range(m), y_true] = 1
        
        delta = self.a_values[-1] - y_one_hot
        
        # Обратный проход
        for i in range(len(self.weights) - 1, -1, -1):
            dW[i] = np.dot(self.a_values[i].T, delta) / m
            db[i] = np.sum(delta, axis=0, keepdims=True) / m
            
            if i > 0:
                delta = np.dot(delta, self.weights[i].T) * self.activation_derivative(self.z_values[i-1])
        
        return dW, db
    
    def update_parameters(self, dW, db):
        """Обновление весов и смещений"""
        for i in range(len(self.weights)):
            self.weights[i] -= self.learning_rate * dW[i]
            self.biases[i] -= self.learning_rate * db[i]
    
    def fit(self, X_train, y_train, X_val=None, y_val=None, epochs=100, batch_size=32, verbose=True):
        """Обучение сети"""
        n_samples = X_train.shape[0]
        
        for epoch in range(epochs):
            # Перемешивание данных
            indices = np.random.permutation(n_samples)
            X_shuffled = X_train[indices]
            y_shuffled = y_train[indices]
            
            # Mini-batch gradient descent
            epoch_loss = 0
            for i in range(0, n_samples, batch_size):
                X_batch = X_shuffled[i:i+batch_size]
                y_batch = y_shuffled[i:i+batch_size]
                
                # Прямое распространение
                y_pred = self.forward_propagation(X_batch)
                
                # Вычисление потерь
                batch_loss = self.compute_loss(y_batch, y_pred)
                epoch_loss += batch_loss
                
                # Обратное распространение
                dW, db = self.backward_propagation(X_batch, y_batch)
                
                # Обновление параметров
                self.update_parameters(dW, db)
            
            # Средняя ошибка за эпоху
            epoch_loss /= (n_samples // batch_size)
            
            # Точность на обучающей выборке
            train_pred = self.predict(X_train)
            train_accuracy = accuracy_score(y_train, train_pred)
            
            self.train_losses.append(epoch_loss)
            self.train_accuracies.append(train_accuracy)
            
            # Валидация
            if X_val is not None and y_val is not None:
                val_pred_proba = self.forward_propagation(X_val)
                val_loss = self.compute_loss(y_val, val_pred_proba)
                val_pred = self.predict(X_val)
                val_accuracy = accuracy_score(y_val, val_pred)
                
                self.val_losses.append(val_loss)
                self.val_accuracies.append(val_accuracy)
                
                if verbose and (epoch + 1) % 10 == 0:
                    print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f} - Acc: {train_accuracy:.4f} - Val Loss: {val_loss:.4f} - Val Acc: {val_accuracy:.4f}")
            else:
                if verbose and (epoch + 1) % 10 == 0:
                    print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f} - Accuracy: {train_accuracy:.4f}")
    
    def predict(self, X):
        """Предсказание классов"""
        y_pred_proba = self.forward_propagation(X)
        return np.argmax(y_pred_proba, axis=1)
    
    def predict_proba(self, X):
        """Предсказание вероятностей"""
        return self.forward_propagation(X)

# ============================================================================
# ЧАСТЬ 2: ПОДГОТОВКА ДАННЫХ
# ============================================================================

In [ ]:
def prepare_text_data(df, max_words=5000, max_len=100):
    """
    Подготовка текстовых данных для нейронных сетей
    """
    print("Подготовка данных...")
    
    # Очистка данных
    df_clean = df.dropna(subset=['review_text', 'mark']).copy()
    df_clean['review_text'] = df_clean['review_text'].astype(str)
    
    # Подготовка целевой переменной
    df_clean['mark'] = df_clean['mark'].astype(int)
    
    # Преобразование меток в числа (0, 1, 2, 3, 4 для оценок 1-5)
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(df_clean['mark'])
    
    print(f"Классы: {label_encoder.classes_}")
    print(f"Распределение классов: {np.bincount(y)}")
    
    # Токенизация текста
    tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
    tokenizer.fit_on_texts(df_clean['review_text'])
    
    # Преобразование в последовательности
    sequences = tokenizer.texts_to_sequences(df_clean['review_text'])
    X_seq = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')
    
    print(f"Размер словаря: {len(tokenizer.word_index)}")
    print(f"Форма данных: {X_seq.shape}")
    
    return X_seq, y, tokenizer, label_encoder


def prepare_tfidf_data(df, max_features=5000):
    """
    Подготовка данных с TF-IDF для MLP
    """
    print("Подготовка TF-IDF данных для MLP...")
    
    df_clean = df.dropna(subset=['review_text', 'mark']).copy()
    df_clean['review_text'] = df_clean['review_text'].astype(str)
    df_clean['mark'] = df_clean['mark'].astype(int)
    
    # TF-IDF векторизация
    vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2))
    X_tfidf = vectorizer.fit_transform(df_clean['review_text']).toarray()
    
    # Целевая переменная
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(df_clean['mark'])
    
    return X_tfidf, y, vectorizer, label_encoder

# ============================================================================
# ЧАСТЬ 3: ПОСТРОЕНИЕ МОДЕЛЕЙ KERAS
# ============================================================================

In [ ]:
def build_rnn_model(input_dim, max_len, num_classes):
    """
    Построение RNN модели
    """
    model = Sequential([
        Embedding(input_dim=input_dim, output_dim=128, input_length=max_len),
        SimpleRNN(64, return_sequences=True),
        Dropout(0.3),
        SimpleRNN(32),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model


def build_lstm_model(input_dim, max_len, num_classes):
    """
    Построение LSTM модели
    """
    model = Sequential([
        Embedding(input_dim=input_dim, output_dim=128, input_length=max_len),
        LSTM(64, return_sequences=True),
        Dropout(0.3),
        LSTM(32),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model


def build_bidirectional_lstm_model(input_dim, max_len, num_classes):
    """
    Построение Bidirectional LSTM модели
    """
    model = Sequential([
        Embedding(input_dim=input_dim, output_dim=128, input_length=max_len),
        Bidirectional(LSTM(64, return_sequences=True)),
        Dropout(0.3),
        Bidirectional(LSTM(32)),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.2),
        Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# ============================================================================
# ЧАСТЬ 4: ОБУЧЕНИЕ И ОЦЕНКА МОДЕЛЕЙ
# ============================================================================

In [ ]:
def train_and_evaluate_models(df_unbalanced, df_balanced):
    """
    Главная функция для обучения и оценки всех моделей
    """
    results = []
    all_histories = {}
    
    # Параметры
    max_words = 5000
    max_len = 100
    epochs = 50
    batch_size = 32
    
    print("="*70)
    print("ОБУЧЕНИЕ МОДЕЛЕЙ НА НЕСБАЛАНСИРОВАННЫХ ДАННЫХ")
    print("="*70)
    
    # --- НЕСБАЛАНСИРОВАННЫЕ ДАННЫЕ ---
    
    # Подготовка данных для RNN/LSTM
    X_seq_unbal, y_unbal, tokenizer_unbal, le_unbal = prepare_text_data(
        df_unbalanced, max_words=max_words, max_len=max_len
    )
    
    X_train_seq_unbal, X_test_seq_unbal, y_train_unbal, y_test_unbal = train_test_split(
        X_seq_unbal, y_unbal, test_size=0.2, random_state=42, stratify=y_unbal
    )
    
    # Подготовка данных для MLP
    X_tfidf_unbal, y_tfidf_unbal, vec_unbal, le_tfidf_unbal = prepare_tfidf_data(
        df_unbalanced, max_features=max_words
    )
    
    X_train_tfidf_unbal, X_test_tfidf_unbal, y_train_tfidf_unbal, y_test_tfidf_unbal = train_test_split(
        X_tfidf_unbal, y_tfidf_unbal, test_size=0.2, random_state=42, stratify=y_tfidf_unbal
    )
    
    # Дополнительная валидационная выборка
    X_train_seq_unbal, X_val_seq_unbal, y_train_seq_unbal, y_val_seq_unbal = train_test_split(
        X_train_seq_unbal, y_train_unbal, test_size=0.2, random_state=42, stratify=y_train_unbal
    )
    
    X_train_tfidf_unbal, X_val_tfidf_unbal, y_train_tfidf_unbal, y_val_tfidf_unbal = train_test_split(
        X_train_tfidf_unbal, y_train_tfidf_unbal, test_size=0.2, random_state=42, stratify=y_train_tfidf_unbal
    )
    
    num_classes = len(np.unique(y_unbal))
    vocab_size = min(max_words, len(tokenizer_unbal.word_index) + 1)
    
    # 1. Самописная MLP на несбалансированных данных
    print("\n1. Обучение самописной MLP (несбалансированные данные)...")
    start_time = time.time()
    
    mlp_custom_unbal = NeuralNetwork(
        layer_sizes=[X_train_tfidf_unbal.shape[1], 128, 64, num_classes],
        learning_rate=0.01,
        activation='relu'
    )
    
    mlp_custom_unbal.fit(
        X_train_tfidf_unbal, y_train_tfidf_unbal,
        X_val_tfidf_unbal, y_val_tfidf_unbal,
        epochs=epochs, batch_size=batch_size, verbose=True
    )
    
    train_time_mlp_unbal = time.time() - start_time
    y_pred_mlp_unbal = mlp_custom_unbal.predict(X_test_tfidf_unbal)
    acc_mlp_unbal = accuracy_score(y_test_tfidf_unbal, y_pred_mlp_unbal)
    
    results.append({
        'Model': 'Custom MLP',
        'Data': 'Unbalanced',
        'Accuracy': acc_mlp_unbal,
        'Training_Time': train_time_mlp_unbal
    })
    
    all_histories['Custom MLP Unbalanced'] = {
        'train_loss': mlp_custom_unbal.train_losses,
        'val_loss': mlp_custom_unbal.val_losses,
        'train_acc': mlp_custom_unbal.train_accuracies,
        'val_acc': mlp_custom_unbal.val_accuracies
    }
    
    print(f"Точность Custom MLP (несбалансированные): {acc_mlp_unbal:.4f}")
    
    # 2. RNN на несбалансированных данных
    print("\n2. Обучение RNN (несбалансированные данные)...")
    
    rnn_unbal = build_rnn_model(vocab_size, max_len, num_classes)
    
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    
    start_time = time.time()
    history_rnn_unbal = rnn_unbal.fit(
        X_train_seq_unbal, y_train_seq_unbal,
        validation_data=(X_val_seq_unbal, y_val_seq_unbal),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=1
    )
    train_time_rnn_unbal = time.time() - start_time
    
    y_pred_rnn_unbal = np.argmax(rnn_unbal.predict(X_test_seq_unbal), axis=1)
    acc_rnn_unbal = accuracy_score(y_test_unbal, y_pred_rnn_unbal)
    
    results.append({
        'Model': 'RNN',
        'Data': 'Unbalanced',
        'Accuracy': acc_rnn_unbal,
        'Training_Time': train_time_rnn_unbal
    })
    
    all_histories['RNN Unbalanced'] = history_rnn_unbal.history
    
    print(f"Точность RNN (несбалансированные): {acc_rnn_unbal:.4f}")
    
    # 3. LSTM на несбалансированных данных
    print("\n3. Обучение LSTM (несбалансированные данные)...")
    
    lstm_unbal = build_lstm_model(vocab_size, max_len, num_classes)
    
    start_time = time.time()
    history_lstm_unbal = lstm_unbal.fit(
        X_train_seq_unbal, y_train_seq_unbal,
        validation_data=(X_val_seq_unbal, y_val_seq_unbal),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=1
    )
    train_time_lstm_unbal = time.time() - start_time
    
    y_pred_lstm_unbal = np.argmax(lstm_unbal.predict(X_test_seq_unbal), axis=1)
    acc_lstm_unbal = accuracy_score(y_test_unbal, y_pred_lstm_unbal)
    
    results.append({
        'Model': 'LSTM',
        'Data': 'Unbalanced',
        'Accuracy': acc_lstm_unbal,
        'Training_Time': train_time_lstm_unbal
    })
    
    all_histories['LSTM Unbalanced'] = history_lstm_unbal.history
    
    print(f"Точность LSTM (несбалансированные): {acc_lstm_unbal:.4f}")
    
    print("\n" + "="*70)
    print("ОБУЧЕНИЕ МОДЕЛЕЙ НА СБАЛАНСИРОВАННЫХ ДАННЫХ")
    print("="*70)
    
    # --- СБАЛАНСИРОВАННЫЕ ДАННЫЕ ---
    
    # Подготовка данных для RNN/LSTM
    X_seq_bal, y_bal, tokenizer_bal, le_bal = prepare_text_data(
        df_balanced, max_words=max_words, max_len=max_len
    )
    
    X_train_seq_bal, X_test_seq_bal, y_train_bal, y_test_bal = train_test_split(
        X_seq_bal, y_bal, test_size=0.2, random_state=42, stratify=y_bal
    )
    
    # Подготовка данных для MLP
    X_tfidf_bal, y_tfidf_bal, vec_bal, le_tfidf_bal = prepare_tfidf_data(
        df_balanced, max_features=max_words
    )
    
    X_train_tfidf_bal, X_test_tfidf_bal, y_train_tfidf_bal, y_test_tfidf_bal = train_test_split(
        X_tfidf_bal, y_tfidf_bal, test_size=0.2, random_state=42, stratify=y_tfidf_bal
    )
    
    # Валидационные выборки
    X_train_seq_bal, X_val_seq_bal, y_train_seq_bal, y_val_seq_bal = train_test_split(
        X_train_seq_bal, y_train_bal, test_size=0.2, random_state=42, stratify=y_train_bal
    )
    
    X_train_tfidf_bal, X_val_tfidf_bal, y_train_tfidf_bal, y_val_tfidf_bal = train_test_split(
        X_train_tfidf_bal, y_train_tfidf_bal, test_size=0.2, random_state=42, stratify=y_train_tfidf_bal
    )
    
    # 4. Самописная MLP на сбалансированных данных
    print("\n4. Обучение самописной MLP (сбалансированные данные)...")
    start_time = time.time()
    
    mlp_custom_bal = NeuralNetwork(
        layer_sizes=[X_train_tfidf_bal.shape[1], 128, 64, num_classes],
        learning_rate=0.01,
        activation='relu'
    )
    
    mlp_custom_bal.fit(
        X_train_tfidf_bal, y_train_tfidf_bal,
        X_val_tfidf_bal, y_val_tfidf_bal,
        epochs=epochs, batch_size=batch_size, verbose=True
    )
    
    train_time_mlp_bal = time.time() - start_time
    y_pred_mlp_bal = mlp_custom_bal.predict(X_test_tfidf_bal)
    acc_mlp_bal = accuracy_score(y_test_tfidf_bal, y_pred_mlp_bal)
    
    results.append({
        'Model': 'Custom MLP',
        'Data': 'Balanced',
        'Accuracy': acc_mlp_bal,
        'Training_Time': train_time_mlp_bal
    })
    
    all_histories['Custom MLP Balanced'] = {
        'train_loss': mlp_custom_bal.train_losses,
        'val_loss': mlp_custom_bal.val_losses,
        'train_acc': mlp_custom_bal.train_accuracies,
        'val_acc': mlp_custom_bal.val_accuracies
    }
    
    print(f"Точность Custom MLP (сбалансированные): {acc_mlp_bal:.4f}")
    
    # 5. RNN на сбалансированных данных
    print("\n5. Обучение RNN (сбалансированные данные)...")
    
    rnn_bal = build_rnn_model(vocab_size, max_len, num_classes)
    
    start_time = time.time()
    history_rnn_bal = rnn_bal.fit(
        X_train_seq_bal, y_train_seq_bal,
        validation_data=(X_val_seq_bal, y_val_seq_bal),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=1
    )
    train_time_rnn_bal = time.time() - start_time
    
    y_pred_rnn_bal = np.argmax(rnn_bal.predict(X_test_seq_bal), axis=1)
    acc_rnn_bal = accuracy_score(y_test_bal, y_pred_rnn_bal)
    
    results.append({
        'Model': 'RNN',
        'Data': 'Balanced',
        'Accuracy': acc_rnn_bal,
        'Training_Time': train_time_rnn_bal
    })
    
    all_histories['RNN Balanced'] = history_rnn_bal.history
    
    print(f"Точность RNN (сбалансированные): {acc_rnn_bal:.4f}")
    
    # 6. LSTM на сбалансированных данных
    print("\n6. Обучение LSTM (сбалансированные данные)...")
    
    lstm_bal = build_lstm_model(vocab_size, max_len, num_classes)
    
    start_time = time.time()
    history_lstm_bal = lstm_bal.fit(
        X_train_seq_bal, y_train_seq_bal,
        validation_data=(X_val_seq_bal, y_val_seq_bal),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=1
    )
    train_time_lstm_bal = time.time() - start_time
    
    y_pred_lstm_bal = np.argmax(lstm_bal.predict(X_test_seq_bal), axis=1)
    acc_lstm_bal = accuracy_score(y_test_bal, y_pred_lstm_bal)
    
    results.append({
        'Model': 'LSTM',
        'Data': 'Balanced',
        'Accuracy': acc_lstm_bal,
        'Training_Time': train_time_lstm_bal
    })
    
    all_histories['LSTM Balanced'] = history_lstm_bal.history
    
    print(f"Точность LSTM (сбалансированные): {acc_lstm_bal:.4f}")
    
    return results, all_histories

# ============================================================================
# ЧАСТЬ 5: ВИЗУАЛИЗАЦИЯ И АНАЛИЗ
# ============================================================================